# Preprocesamiento — Heart Disease Dataset

Proyecto académico: detección temprana de enfermedades cardiovasculares mediante clasificación binaria.

> Dataset: UCI Heart Disease Dataset, ID=45. El dataset se carga directamente con `ucimlrepo`, sin CSV ni Google Drive.

## 1. Instalación e importación de librerías

In [1]:
!pip -q install ucimlrepo

import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2. Descarga y preparación inicial del dataset

In [2]:
heart_disease = fetch_ucirepo(id=45)

features = heart_disease.data.features
targets = heart_disease.data.targets

df = pd.concat([features, targets], axis=1)
df["target"] = df["num"].apply(lambda value: 0 if value == 0 else 1)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0,0


### Separación de predictores y variable objetivo

La columna `num` codifica directamente el diagnóstico original. Por eso se elimina de las variables predictoras para evitar *data leakage*. La variable `target` se mantiene como objetivo binario.

In [3]:
X_raw = df.drop(columns=["num", "target"])
y = df["target"].copy()

print("Forma de X_raw:", X_raw.shape)
print("Forma de y:", y.shape)

Forma de X_raw: (303, 13)
Forma de y: (303,)


## 3. Tratamiento de valores faltantes

Según el EDA, existen 6 valores faltantes concentrados en `ca` y `thal`.

In [4]:
missing_values = X_raw.isna().sum()
missing_values[missing_values > 0]

,0
ca,4
thal,2


In [5]:
def imputar_valores_faltantes(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Imputa valores faltantes usando reglas clínicas simples y reproducibles.

    Parámetros
    ----------
    dataframe : pd.DataFrame
        Variables predictoras originales del Heart Disease Dataset.

    Retorna
    -------
    pd.DataFrame
        Copia del dataset sin valores faltantes en `ca` y `thal`.
    """
    dataframe_imputado = dataframe.copy()

    # `ca` es una variable cuantitativa discreta. La mediana es robusta ante
    # valores extremos y conserva una medida central adecuada para muestras pequeñas.
    dataframe_imputado["ca"] = dataframe_imputado["ca"].fillna(
        dataframe_imputado["ca"].median()
    )

    # `thal` es categórica. La moda conserva la categoría observada más frecuente
    # sin inventar una categoría artificial para solo unos pocos casos faltantes.
    dataframe_imputado["thal"] = dataframe_imputado["thal"].fillna(
        dataframe_imputado["thal"].mode()[0]
    )

    return dataframe_imputado


X_imputed = imputar_valores_faltantes(X_raw)
X_imputed.isna().sum().sum()

np.int64(0)

## 4. Identificación de variables numéricas y categóricas

In [6]:
NUMERIC_COLUMNS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
CATEGORICAL_COLUMNS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

missing_expected_columns = set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS) - set(X_imputed.columns)
if missing_expected_columns:
    raise ValueError(f"Faltan columnas esperadas: {missing_expected_columns}")

print("Variables numéricas:", NUMERIC_COLUMNS)
print("Variables categóricas:", CATEGORICAL_COLUMNS)

Variables numéricas: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Variables categóricas: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']


## 5. Codificación de variables categóricas

Se aplica One-Hot Encoding porque los modelos de Machine Learning trabajan con entradas numéricas. Además, esta técnica evita imponer un orden artificial entre categorías clínicas como `cp`, `slope` o `thal`.

In [8]:
def codificar_categoricas(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Aplica One-Hot Encoding sobre las variables categóricas esperadas.

    Parámetros
    ----------
    dataframe : pd.DataFrame
        Dataset con variables clínicas imputadas.

    Retorna
    -------
    pd.DataFrame
        Dataset con variables categóricas transformadas en columnas binarias.
    """
    return pd.get_dummies(
        dataframe,
        columns=CATEGORICAL_COLUMNS,
        drop_first=False,
        dtype=int,
    )


X_encoded = codificar_categoricas(X_imputed)
X_encoded.head()

,age,trestbps,chol,thalach,oldpeak,ca,sex_0,sex_1,cp_1,cp_2,cp_3,cp_4,fbs_0,fbs_1,restecg_0,restecg_1,restecg_2,exang_0,exang_1,slope_1,slope_2,slope_3,thal_3.0,thal_6.0,thal_7.0
0,63,145,233,150,2.3,0.0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,0,1,0,1,0
1,67,160,286,108,1.5,3.0,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,1,0,0
2,67,120,229,129,2.6,2.0,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,0,0,1
3,37,130,250,187,3.5,0.0,0,1,0,0,1,0,1,0,1,0,0,1,0,0,0,1,1,0,0
4,41,130,204,172,1.4,0.0,1,0,0,1,0,0,1,0,0,0,1,1,0,1,0,0,1,0,0


## 6. Escalamiento de variables numéricas

Se usa `StandardScaler` únicamente sobre variables numéricas. Esto es importante para SVM, Regresión Logística y Redes Neuronales porque estos modelos son sensibles a la escala de las variables: una variable con magnitudes grandes puede dominar el proceso de optimización o el cálculo de distancias.

In [9]:
def escalar_numericas(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Estandariza solo las variables numéricas del dataset.

    Parámetros
    ----------
    dataframe : pd.DataFrame
        Dataset con variables categóricas ya codificadas.

    Retorna
    -------
    pd.DataFrame
        Dataset final con variables numéricas escaladas y variables binarias intactas.
    """
    dataframe_scaled = dataframe.copy()
    scaler = StandardScaler()
    dataframe_scaled[NUMERIC_COLUMNS] = scaler.fit_transform(dataframe_scaled[NUMERIC_COLUMNS])
    return dataframe_scaled


X = escalar_numericas(X_encoded)
X.head()

,age,trestbps,chol,thalach,oldpeak,ca,sex_0,sex_1,cp_1,cp_2,cp_3,cp_4,fbs_0,fbs_1,restecg_0,restecg_1,restecg_2,exang_0,exang_1,slope_1,slope_2,slope_3,thal_3.0,thal_6.0,thal_7.0
0,0.948726,0.757525,-0.264900,0.017197,1.087338,-0.711131,0,1,1,0,0,0,0,1,0,0,1,1,0,0,0,1,0,1,0
1,1.392002,1.611220,0.760415,-1.821905,0.397182,2.504881,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,1,0,0
2,1.392002,-0.665300,-0.342283,-0.902354,1.346147,1.432877,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,0,0,1
3,-1.932564,-0.096170,0.063974,1.637359,2.122573,-0.711131,0,1,0,0,1,0,1,0,1,0,0,1,0,0,0,1,1,0,0
4,-1.489288,-0.096170,-0.825922,0.980537,0.310912,-0.711131,1,0,0,1,0,0,1,0,0,0,1,1,0,1,0,0,1,0,0


## 7. Construcción del dataset final

In [10]:
print("Número final de características:", X.shape[1])
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Número final de características: 25
Forma de X: (303, 25)
Forma de y: (303,)


## 8. División del dataset

Se utiliza una división estratificada para conservar la proporción de clases en entrenamiento y prueba. Esto es relevante porque la prevalencia de enfermedad es aproximadamente 45.9%.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Tamaño del conjunto de entrenamiento:", X_train.shape[0])
print("Tamaño del conjunto de prueba:", X_test.shape[0])

class_distribution = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).rename(index={0: "Sin enfermedad", 1: "Con enfermedad"})

class_distribution

Tamaño del conjunto de entrenamiento: 242
Tamaño del conjunto de prueba: 61


,train,test
target,,
Sin enfermedad,0.541322,0.540984
Con enfermedad,0.458678,0.459016


## 9. Validaciones del dataset final

In [12]:
def validar_dataset_ml(features_dataframe: pd.DataFrame) -> None:
    """Valida que el dataset final sea apto para entrenamiento de modelos.

    Parámetros
    ----------
    features_dataframe : pd.DataFrame
        Matriz final de características después del preprocesamiento.

    Retorna
    -------
    None
        La función falla con `AssertionError` si alguna validación no se cumple.
    """
    assert features_dataframe.isna().sum().sum() == 0, "Existen valores nulos."
    assert len(features_dataframe.select_dtypes(exclude="number").columns) == 0, (
        "Existen variables no numéricas."
    )


validar_dataset_ml(X)
print("Validación correcta: no hay nulos ni variables no numéricas.")
X.head()

Validación correcta: no hay nulos ni variables no numéricas.


,age,trestbps,chol,thalach,oldpeak,ca,sex_0,sex_1,cp_1,cp_2,cp_3,cp_4,fbs_0,fbs_1,restecg_0,restecg_1,restecg_2,exang_0,exang_1,slope_1,slope_2,slope_3,thal_3.0,thal_6.0,thal_7.0
0,0.948726,0.757525,-0.264900,0.017197,1.087338,-0.711131,0,1,1,0,0,0,0,1,0,0,1,1,0,0,0,1,0,1,0
1,1.392002,1.611220,0.760415,-1.821905,0.397182,2.504881,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,1,0,0
2,1.392002,-0.665300,-0.342283,-0.902354,1.346147,1.432877,0,1,0,0,0,1,1,0,0,0,1,0,1,0,1,0,0,0,1
3,-1.932564,-0.096170,0.063974,1.637359,2.122573,-0.711131,0,1,0,0,1,0,1,0,1,0,0,1,0,0,0,1,1,0,0
4,-1.489288,-0.096170,-0.825922,0.980537,0.310912,-0.711131,1,0,0,1,0,0,1,0,0,0,1,1,0,1,0,0,1,0,0


## 10. Guardado opcional del dataset procesado

In [13]:
processed_df = X.copy()
processed_df["target"] = y.values

processed_df.to_csv("heart_disease_processed.csv", index=False)
print("Archivo guardado: heart_disease_processed.csv")

Archivo guardado: heart_disease_processed.csv


## 11. Conclusiones automáticas del preprocesamiento

In [14]:
def construir_conclusiones_preprocesamiento(
    features_dataframe: pd.DataFrame,
    target_series: pd.Series,
    train_target: pd.Series,
    test_target: pd.Series,
) -> list[str]:
    """Construye conclusiones automáticas sobre el preprocesamiento realizado.

    Parámetros
    ----------
    features_dataframe : pd.DataFrame
        Matriz final de características lista para modelos de Machine Learning.
    target_series : pd.Series
        Variable objetivo binaria completa.
    train_target : pd.Series
        Variable objetivo del conjunto de entrenamiento.
    test_target : pd.Series
        Variable objetivo del conjunto de prueba.

    Retorna
    -------
    list[str]
        Conclusiones breves para documentar el preprocesamiento en el artículo.
    """
    null_count = int(features_dataframe.isna().sum().sum())
    non_numeric_count = len(features_dataframe.select_dtypes(exclude="number").columns)
    disease_rate = target_series.mean() * 100
    train_rate = train_target.mean() * 100
    test_rate = test_target.mean() * 100

    return [
        f"El dataset final contiene {features_dataframe.shape[1]} variables predictoras numéricas.",
        f"El preprocesamiento eliminó valores faltantes: nulos restantes = {null_count}.",
        f"No quedan variables no numéricas: cantidad = {non_numeric_count}.",
        f"La prevalencia global de enfermedad es {disease_rate:.1f}%.",
        f"La división estratificada conservó la distribución: train={train_rate:.1f}%, test={test_rate:.1f}%.",
        "El dataset queda listo para comparar modelos interpretables, Deep Learning y explicabilidad con SHAP.",
    ]


for index, conclusion in enumerate(
    construir_conclusiones_preprocesamiento(X, y, y_train, y_test),
    start=1,
):
    print(f"{index}. {conclusion}")

1. El dataset final contiene 25 variables predictoras numéricas.
2. El preprocesamiento eliminó valores faltantes: nulos restantes = 0.
3. No quedan variables no numéricas: cantidad = 0.
4. La prevalencia global de enfermedad es 45.9%.
5. La división estratificada conservó la distribución: train=45.9%, test=45.9%.
6. El dataset queda listo para comparar modelos interpretables, Deep Learning y explicabilidad con SHAP.
